In [1]:
SCOPETYPE = 'CWNANO'
PLATFORM = 'CWNANO'
CRYPTO_TARGET='SHUFFLEDAES' 
SS_VER='SS_VER_2_1'

In [2]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍


In [3]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"
cd ../../firmware/mcu/simpleserial-aes
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

Building for platform CWNANO with CRYPTO_TARGET=SHUFFLEDAES
SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
Blank crypto options, building for AES128
arm-none-eabi-gcc (15:14.2.rel1-1) 14.2.1 20241119
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWNANO 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
.
.
.
Compiling:
Compiling:
Compiling:
Compiling:
.
Compiling:
Compiling:
Assembling: .././hal//stm32f0/stm32f0_startup.S
arm-none-eabi-gcc -c -mcpu=cortex-m0 -I. -x assembler-with-cpp -mthumb -mfloat-abi=soft -ffunction-sections -DF_CPU=7372800 -Wa,-gstabs,-adhlns=objdir-CWNANO/stm32f0_startup.lst -I.././simpleserial/ -I.././hal/ -I.././hal/ -I.././hal//stm32f0 -I.././hal//stm32f0/CMSIS -I.././hal//stm32f0/CMSIS/core -I.././hal//stm32f0/CMSIS/device -I.././hal//stm32f0/Legacy -I.././simpleserial/ -

In [4]:
cw.program_target(scope, prog, "../../firmware/mcu/simpleserial-aes/simpleserial-aes-{}.hex".format(PLATFORM))

Detected known STMF32: STM32F04xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 6643 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 6643 bytes


In [5]:
proj = cw.create_project("Lab 4_3", overwrite=True)

In [6]:
from tqdm.notebook import trange
import numpy as np
import time

ktp = cw.ktp.Basic()
trace_array = []
textin_array = []

N = 10000
for i in trange(N, desc='Capturing traces'):
    key, text = ktp.next()
    trace = cw.capture_trace(scope, target, text, key)
    if not trace:
        continue
    
    proj.traces.append(trace)

Capturing traces:   0%|          | 0/10000 [00:00<?, ?it/s]

In [7]:
scope.dis()
target.dis()

In [8]:
# we can access wave, textin, etc as a whole with proj.traces
for trace in proj.traces:
    print(trace.wave[0], trace.textin, trace.textout, trace.key)

# can also access individually with proj.waves, proj.textins, etc.
for wave in proj.waves:
    print(wave[0])

-0.03515625 CWbytearray(b'bb 05 b0 a0 12 2c 6b fe 7a 68 90 86 47 af 13 fe') CWbytearray(b'e6 13 75 f1 2d cd f8 71 2f ae a3 c3 59 2f 5f bc') CWbytearray(b'2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c')
0.06640625 CWbytearray(b'9f f2 36 3e 24 9a 80 e7 7b 10 5d 21 b2 e9 33 e1') CWbytearray(b'92 3a c6 ca c4 73 78 b6 95 3d b2 cc f5 7c 75 3d') CWbytearray(b'2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c')
-0.05078125 CWbytearray(b'3f 3a f3 c1 77 68 b2 2f 3d df 39 8f 0c 18 d3 06') CWbytearray(b'cc 04 a3 ba 7f bf 3e 5b 85 5e fc f3 4a 26 6a 2e') CWbytearray(b'2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c')
0.12109375 CWbytearray(b'd8 8b 45 d8 a9 bb cf 46 fd 92 bc 62 0a 3b a2 4f') CWbytearray(b'b2 a2 6d 22 44 56 ab a6 4b 5f 05 04 0e 76 20 aa') CWbytearray(b'2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c')
-0.08203125 CWbytearray(b'db c6 fd 3b 2a 72 db 4c 3b 03 40 99 ce f8 34 42') CWbytearray(b'f6 68 4d 02 5a 35 b6 9b 15 d2 87 67 d6 6f 85 31') CWbytearray(b'2b 7e 15 16 28 ae d2 a6 ab f7 15

In [9]:
import chipwhisperer.analyzer as cwa

In [10]:
leak_model = cwa.leakage_models.sbox_output

In [11]:
attack = cwa.cpa(proj, leak_model)

In [12]:
print(attack)

project     = <chipwhisperer.common.api.ProjectFormat.Project object at 0x000002312434FBF0>
leak_model  = <chipwhisperer.analyzer.attacks.models.AES128_8bit.AES128_8bit object at 0x00000231553B5D90>
algorithm   = <chipwhisperer.analyzer.attacks.cpa_algorithms.progressive.CPAProgressive object at 0x0000023117CBB890>
trace_range = [0, 10000]
point_range = [0, 5000]
subkey_list = range(0, 16)



In [13]:
results = attack.run()

In [14]:
print(results)

Subkey KGuess Correlation
  00    0xF4    0.15590
  01    0x2B    0.13908
  02    0x09    0.14826
  03    0xF4    0.13972
  04    0x0B    0.15987
  05    0x0B    0.13686
  06    0x0F    0.17291
  07    0x0F    0.14637
  08    0xF0    0.16125
  09    0x4F    0.12493
  10    0xF0    0.15328
  11    0xF4    0.14535
  12    0x0B    0.15891
  13    0x0B    0.13390
  14    0xF6    0.16533
  15    0x09    0.14118



In [15]:
import pandas as pd
stat_data = results.find_maximums()
df = pd.DataFrame(stat_data).transpose()
print(df.head())

                               0                               1   \
0  [244, 50, 0.15589814098554036]   [43, 60, 0.13907940505950425]   
1   [79, 50, 0.14995922559387662]  [240, 60, 0.13530255740877659]   
2     [15, 50, 0.143582830066852]   [246, 59, 0.1316438745522138]   
3    [9, 50, 0.14276948436039374]  [244, 59, 0.13097769813865878]   
4  [240, 50, 0.14034342531293698]    [9, 60, 0.13087722139413183]   

                               2                               3   \
0    [9, 68, 0.14826423617409898]  [244, 78, 0.13972154333627695]   
1  [242, 68, 0.14645485372943054]   [15, 78, 0.13549971212362344]   
2  [244, 68, 0.14471739404026093]    [180, 78, 0.135224460362031]   
3   [246, 68, 0.1441085204391323]  [242, 77, 0.13456262204687397]   
4   [15, 68, 0.14408831149348486]   [79, 78, 0.12413150713419245]   

                               4                                5   \
0   [11, 86, 0.15987203619521403]     [11, 95, 0.1368587300018319]   
1   [15, 86, 0.154161237393003

In [16]:
key = proj.keys[0]
def format_stat(stat):
    return str("{:02X}<br>{:.3f}".format(stat[0], stat[2]))

def color_corr_key(row):
    global key
    ret = [""] * 16
    for i,bnum in enumerate(row):
        if bnum[0] == key[i]:
            ret[i] = "color: red"
        else:
            ret[i] = ""
    return ret

df.head().style.format(format_stat).apply(color_corr_key, axis=1)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,F40.156,2B0.139,090.148,F40.140,0B0.160,0B0.137,0F0.173,0F0.146,F00.161,4F0.125,F00.153,F40.145,0B0.159,0B0.134,F60.165,090.141
1,4F0.150,F00.135,F20.146,0F0.135,0F0.154,4F0.136,F00.158,0B0.139,0D0.144,0F0.124,4D0.150,0F0.133,0F0.154,B00.128,F00.157,0F0.140
2,0F0.144,F60.132,F40.145,B40.135,F60.153,B00.136,B60.157,F60.128,4B0.138,F40.124,090.145,0B0.127,F40.141,F00.126,0F0.155,0D0.134
3,090.143,F40.131,F60.144,F20.135,F00.149,F20.131,0D0.155,F40.126,F20.136,0D0.119,F60.142,090.127,090.138,0D0.124,0B0.152,F00.133
4,F00.140,090.131,0F0.144,4F0.124,090.148,BC0.128,B40.154,B00.124,0F0.133,D40.118,0B0.141,F00.124,B00.136,0F0.122,F40.148,F40.132


In [ ]:
from IPython.display import clear_output
import numpy as np
        
def stats_callback():
    results = attack.results
    results.set_known_key(key)
    stat_data = results.find_maximums()
    df = pd.DataFrame(stat_data).transpose()
    clear_output(wait=True)
    display(df.head().style.format(format_stat).apply(color_corr_key,axis=1))
    
results = attack.run(stats_callback, 10)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,F40.170,2B0.144,0F0.150,F40.148,0B0.175,0B0.143,0F0.169,0F0.149,F00.162,F40.129,F00.159,F40.140,0B0.159,0D0.134,F60.163,F40.147
1,4F0.153,F40.143,F20.145,0F0.137,0F0.162,4F0.141,0B0.166,6B0.134,0F0.145,4F0.128,4D0.147,0F0.135,0F0.154,0B0.132,0F0.155,0D0.140
2,090.149,0B0.141,F60.145,B40.134,F40.160,B00.140,F00.159,F00.131,0D0.142,2B0.122,0B0.141,090.132,4B0.140,F00.127,0B0.153,0F0.139
3,B00.147,090.140,D20.144,F00.133,090.157,F20.139,B60.155,B00.131,4B0.141,0D0.122,B00.140,F20.126,B20.135,0F0.126,F40.151,F00.137
4,0F0.142,F00.139,4B0.143,F20.132,F60.149,F40.129,B00.149,F60.131,F20.137,0F0.120,0F0.137,490.124,F40.134,B00.119,F00.149,490.135


In [ ]:
import chipwhisperer as cw
cb = cwa.get_jupyter_callback(attack)
results = attack.run(cb, 10)

In [ ]:
plot_data = cwa.analyzer_plots(results)

In [ ]:
def byte_to_color(idx):
    return hv.Palette.colormaps['Category20'](idx/16.0)

import holoviews as hv
from holoviews.operation.datashader import datashade, shade, dynspread, rasterize
from holoviews.operation import decimate
import pandas as pd, numpy as np

a = []
b = []
hv.extension('bokeh')
for i in range(0, 16):
    data = plot_data.output_vs_time(i)
    a.append(np.array(data[1]))
    b.append(np.array(data[2]))
    b.append(np.array(data[3]))
    
pda = pd.DataFrame(a).transpose().rename(str, axis='columns')
pdb = pd.DataFrame(b).transpose().rename(str, axis='columns')
curve = hv.Curve(pdb['0'], "Sample").options(color='black')
for i in range(1, 16):
    curve *= hv.Curve(pdb[str(i)]).options(color='black')
    
for i in range(0, 16):
    curve *= hv.Curve(pda[str(i)]).options(color=byte_to_color(i))
decimate(curve.opts(width=900, height=600))

In [ ]:
ret = plot_data.pge_vs_trace(0)
curve = hv.Curve((ret[0],ret[1]), "Traces Used in Calculation", "Partial Guessing Entrop of Byte")
for bnum in range(1, 16):
    ret = plot_data.pge_vs_trace(bnum)
    curve *= hv.Curve((ret[0],ret[1])).opts(color=byte_to_color(bnum))
curve.opts(width=900, height=600)

In [ ]:
a = []
b = []
for bnum in range(0, 16):
    data = plot_data.corr_vs_trace(bnum)
    best = [0] * len(data[1][0])
    for i in range(256):
        if i == key[bnum]:
            a.append(np.array(data[1][i]))
        else:
            if max(best) < max(data[1][i]): best = data[1][i]
    b.append(np.array(best))

pda = pd.DataFrame(a).transpose().rename(str, axis='columns')
pdb = pd.DataFrame(b).transpose().rename(str, axis='columns')
curve = hv.Curve(pdb['0'].tolist(), "Iteration Number", "Max Correlation").options(color='black')
for i in range(1,len(pdb.columns)):
    curve *= hv.Curve(pdb[str(i)]).options(color='black')
    
for i in range(len(pda.columns)):
    curve *= hv.Curve(pda[str(i)]).options(color=byte_to_color(i))
            
curve.opts(width=900, height=600)